In [ ]:
# %%
import numpy as np
from readlif.reader import LifFile
from skimage import io
import os


# %%
def normalize_stack_preserve_decay(stack, method='percentile', percentile=99.5):
    """
    Normalize a z-stack to consistent intensity ranges while preserving
    the natural decay pattern across z-slices.

    Parameters:
    -----------
    stack : ndarray
        3D array (z, y, x)
    method : str
        'percentile' - normalize to percentile across all slices
        'mean' - normalize to mean intensity
        'median' - normalize to median intensity
    percentile : float
        Percentile value to use if method='percentile'

    Returns:
    --------
    normalized_stack : ndarray
        Normalized stack preserving relative intensity decay
    global_norm : float
        The normalization factor used
    """

    # Calculate global normalization factor across entire stack
    if method == 'percentile':
        global_norm = np.percentile(stack, percentile)
    elif method == 'mean':
        global_norm = np.mean(stack[stack > 0])  # Exclude background
    elif method == 'median':
        global_norm = np.median(stack[stack > 0])
    else:
        raise ValueError(f"Unknown method: {method}")

    # Normalize entire stack by global factor
    normalized = stack.astype(np.float32) / global_norm

    return normalized, global_norm


# %%
def save_normalized_stack(normalized_stack, output_path, bit_depth=16, voxel_size=None):
    """
    Save normalized stack as TIFF with appropriate bit depth and metadata.

    Parameters:
    -----------
    normalized_stack : ndarray
        Normalized stack (values typically 0-1)
    output_path : str
        Path to save TIFF
    bit_depth : int
        8 or 16 bit output
    voxel_size : dict or None
        Dictionary with 'x', 'y', 'z' keys for voxel dimensions in micrometers
    """

    if bit_depth == 16:
        max_val = 65535
        dtype = np.uint16
    elif bit_depth == 8:
        max_val = 255
        dtype = np.uint8
    else:
        raise ValueError("bit_depth must be 8 or 16")

    # Scale to bit depth range
    output = np.clip(normalized_stack * max_val, 0, max_val).astype(dtype)

    # Prepare metadata for TIFF
    metadata = {}
    if voxel_size is not None:
        # ImageJ-compatible metadata
        # Resolution is pixels per unit (inverse of voxel size)
        # TIFF uses pixels per cm by default, convert to pixels per µm
        resolution_x = 1.0 / voxel_size['x'] if voxel_size['x'] > 0 else 1.0
        resolution_y = 1.0 / voxel_size['y'] if voxel_size['y'] > 0 else 1.0

        # For ImageJ compatibility, store spacing
        metadata = {
            'spacing': voxel_size['z'],  # Z-spacing in µm
            'unit': 'um',  # micrometers
            'axes': 'ZYX'
        }

        # Save with resolution information
        io.imsave(
            output_path,
            output,
            check_contrast=False,
            resolution=(resolution_x, resolution_y),
            metadata=metadata
        )
    else:
        io.imsave(output_path, output, check_contrast=False)

    print(f"Saved: {os.path.basename(output_path)}")


# %%
def normalize_channel2_from_lif(lif_path, image_index=0,
                                method='percentile', percentile=99.5,
                                save_output=True, output_path=None):
    """
    Load a .lif file and normalize only channel 2 for a specific image.
    """

    lif = LifFile(lif_path)
    img = lif.get_image(image_index)

    # Get dimensions
    n_channels = img.channels
    n_z = img.dims.z
    n_y = img.dims.y
    n_x = img.dims.x

    # Verify channel 2 exists (0-indexed, so channel 2 is the 3rd channel)
    if n_channels < 3:
        raise ValueError(f"Image only has {n_channels} channels. Channel 2 (index 2) does not exist.")

    # Load stack for channel 2 (index 2)
    stack = []
    for z in range(n_z):
        frame = img.get_frame(z=z, t=0, c=1)
        stack.append(frame)

    stack = np.array(stack)

    # Normalize
    normalized, norm_factor = normalize_stack_preserve_decay(
        stack, method=method, percentile=percentile
    )

    metadata = {
        'image_name': img.name,
        'image_index': image_index,
        'n_channels': n_channels,
        'n_z': n_z,
        'n_y': n_y,
        'n_x': n_x,
        'norm_factor': norm_factor,
        'method': method,
        'channel': 2,
        'percentile': percentile if method == 'percentile' else None,
        'original_min': stack.min(),
        'original_max': stack.max(),
        'normalized_min': normalized.min(),
        'normalized_max': normalized.max()
    }

    # Save if requested
    if save_output:
        if output_path is None:
            # Auto-generate output path with image name in ch2normalized folder
            base_dir = os.path.dirname(lif_path)
            lif_basename = os.path.splitext(os.path.basename(lif_path))[0]
            output_dir = os.path.join(base_dir, f"{lif_basename}_ch2normalized")
            os.makedirs(output_dir, exist_ok=True)

            # Sanitize image name for filename
            safe_name = "".join(c for c in img.name if c.isalnum() or c in (' ', '-', '_')).strip()
            safe_name = safe_name.replace(' ', '_')
            output_path = os.path.join(output_dir, f"img{image_index:03d}_{safe_name}.tif")

        save_normalized_stack(normalized, output_path, bit_depth=16)

    return normalized, metadata


# %%
# ---- Batch normalization ----
def normalize_all_images_channel2(lif_path, method='percentile', percentile=99.5,
                                   save_output=True):
    """
    Normalize channel 2 for all images in a .lif file.
    Saves to a folder named: <lif_filename>_ch2normalized

    Parameters:
    -----------
    lif_path : str
        Path to .lif file
    method : str
        Normalization method
    percentile : float
        Percentile for normalization
    save_output : bool
        Whether to save normalized stacks

    Returns:
    --------
    results : list of dict
        List of metadata dictionaries for each processed image
    output_dir : str
        Path to the output directory where files were saved
    """

    lif = LifFile(lif_path)
    n_images = lif.num_images

    # Create output directory
    base_dir = os.path.dirname(lif_path)
    if not base_dir:
        base_dir = "."
    lif_basename = os.path.splitext(os.path.basename(lif_path))[0]
    output_dir = os.path.join(base_dir, f"{lif_basename}_ch2normalized_test")

    if save_output:
        os.makedirs(output_dir, exist_ok=True)

    print(f"Found {n_images} images -> {output_dir}")

    # Process each image
    results = []
    for i in range(n_images):
        try:
            # Generate output path
            img = lif.get_image(i)
            safe_name = "".join(c for c in img.name if c.isalnum() or c in (' ', '-', '_')).strip()
            safe_name = safe_name.replace(' ', '_')
            output_path = os.path.join(output_dir, f"img{i:03d}_{safe_name}.tif")

            normalized, metadata = normalize_channel2_from_lif(
                lif_path,
                image_index=i,
                method=method,
                percentile=percentile,
                save_output=save_output,
                output_path=output_path
            )

            results.append(metadata)

        except Exception as e:
            print(f"ERROR processing image {i}: {e}")
            continue

    print(f"Done: {len(results)}/{n_images} normalized")

    return results, output_dir


# %%
# ---- Set path ----
lif_path = "C:/pathToLif/file.lif"


# %%
# ---- Process ALL images ----
# saves to <lif_filename>_ch2normalized/ folder
results, output_dir = normalize_all_images_channel2(
    lif_path,
    method='percentile',
    percentile=100,
    save_output=True
)

